# Logit Lens — Memory vs. Context Conflict Probe

Layer-by-layer probability probe for context (doc) vs. parametric memory conflict in API tool synthesis.

For each mutation pair `(original, mutated)`, runs a forward pass with `output_hidden_states=True`,
projects each layer's residual stream through the unembedding matrix W_U, and plots
P(original) vs P(mutated) at every layer depth to find where the model commits.

In [ ]:

# ── Check dependencies (pre-installed on H100 clusters) ──────────────────────
import torch, transformers, matplotlib
print(f"torch       {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
# ── Config — edit these ───────────────────────────────────────────────────────
MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"   # or 14B / 72B if you have the VRAM
DATASET    = "zulip"                       # stripe | zulip | alphavantage | jira | ebay_sell
PROBE_AT   = "pre"                        # "pre" = token just before param name; "at" = first token
MAX_TOKENS = 400
WINDOW     = 512   # tokens fed into hidden-states pass (centred on probe position)

# Inline dataset configs (no repo import needed)
# doc_text and mutations_json must be pasted/uploaded below OR fetched from the repo.
# Set RAW_DOC and RAW_MUTATIONS to None to paste inline; or point to local paths.
DOC_PATH        = None   # e.g. "/path/to/docs_mutated/api_get-messages.md"
MUTATIONS_PATH  = None   # e.g. "/path/to/mutations.json"

ENDPOINT_HEADER = {
    "stripe":      "# Create a PaymentIntent",
    "zulip":       "# Get messages\n",
    "alphavantage":"#### TIME_SERIES_DAILY\n",
    "jira":        "## Search for issues using JQL enhanced search (GET)",
    "ebay_sell":   "# getFulfillmentPolicies",
}[DATASET]

print(f'Dataset: {DATASET}  Model: {MODEL_ID}  Probe: {PROBE_AT}')

In [ ]:
# ── Paste doc text here if DOC_PATH is None ───────────────────────────────────
# Copy the relevant mutated doc file content between the triple-quotes.
# For zulip: benchmark/datasets/zulip/docs_mutated/api_get-messages.md

RAW_DOC = """PASTE MUTATED DOC CONTENT HERE"""

# ── Paste mutations JSON here if MUTATIONS_PATH is None ───────────────────────
# Copy benchmark/datasets/<dataset>/mutations.json between the triple-quotes.

RAW_MUTATIONS = """
PASTE MUTATIONS JSON HERE
"""

# If you uploaded the files, set paths above and the next cell will read them.

In [ ]:
import json, math, textwrap
from pathlib import Path

# ── Load doc & mutations ──────────────────────────────────────────────────────
if DOC_PATH:
    raw_doc = Path(DOC_PATH).read_text()
else:
    raw_doc = RAW_DOC

if MUTATIONS_PATH:
    mutations = json.loads(Path(MUTATIONS_PATH).read_text())["mutations"]
else:
    mutations = json.loads(RAW_MUTATIONS)["mutations"]

# ── Extract relevant doc section ──────────────────────────────────────────────
def extract_doc_section(text: str, header: str, char_limit: int = 6000) -> str:
    start = text.rfind(header)
    if start == -1:
        return text[:char_limit]
    level = len(header) - len(header.lstrip("#"))
    next_h = text.find("\n" + "#" * level + " ", start + len(header))
    end = next_h if next_h != -1 else len(text)
    return text[start:end].strip()[:char_limit]

doc_snippet = extract_doc_section(raw_doc, ENDPOINT_HEADER)
print(f"Doc section: {len(doc_snippet)} chars")
print(f"Mutations  : {[m['original']+'→'+m['mutated'] for m in mutations]}")
print()
print(doc_snippet[:500], "...")

In [ ]:
# ── Prompt templates ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are an expert Python developer building MCP (Model Context Protocol) tool servers.
Given an API documentation snippet, write a single Python function that wraps the described endpoint.

Rules:
- Use @mcp.tool() decorator
- Function parameters must exactly match the parameter names documented (spelling matters)
- Use requests to make the HTTP call
- Pass parameters as a dict called `params` for query params or `data` for form-encoded bodies
- Return the JSON response as a dict
- Keep it concise — no docstring needed, just the function
"""

HUMAN_TEMPLATE = """\
Here is the API documentation for one endpoint:

---
{doc_snippet}
---

Write a single Python @mcp.tool() function that implements this endpoint.
Parameter names in your function signature MUST match the documentation exactly.
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": HUMAN_TEMPLATE.format(doc_snippet=doc_snippet)},
]
print('Prompt constructed.')

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
import os, site, sys

# Cache models to fast local NVMe instead of slow NFS home
os.environ.setdefault("HF_HOME", "/data/ebay-rno-h100/notebooks/tstereciu/.hf_cache")

# Include user site-packages in case transformers was pip-installed to ~/.local
_user_site = site.getusersitepackages()
if _user_site not in sys.path:
    sys.path.insert(0, _user_site)

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if device.type == "cuda" else torch.float32   # bfloat16 is native on H100
print(f"Device: {device}  dtype: {dtype}")
if device.type == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.0f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
n_layers   = model.config.num_hidden_layers
hidden_dim = model.config.hidden_size
n_params   = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nLoaded {MODEL_ID}")
print(f"  {n_params:.1f}B params | {n_layers} layers | hidden_dim={hidden_dim}")

In [ ]:
# ── Tokenize prompt & generate response ───────────────────────────────────────
prompt_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
)
if hasattr(prompt_ids, "input_ids"):
    prompt_ids = prompt_ids.input_ids
prompt_ids = prompt_ids.to(device)
print(f"Prompt: {prompt_ids.shape[1]} tokens — generating up to {MAX_TOKENS} more...")

with torch.no_grad():
    gen_ids = model.generate(
        prompt_ids,
        max_new_tokens=MAX_TOKENS,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

response_start = prompt_ids.shape[1]
response_ids   = gen_ids[0, response_start:]
response_text  = tokenizer.decode(response_ids, skip_special_tokens=True)
full_ids       = gen_ids[0].tolist()

print(f"Response ({len(response_ids)} tokens):")
print(textwrap.indent(response_text, "  "))

In [ ]:
# ── Helper: resolve first sub-token ID for each name ─────────────────────────
def get_first_token_ids(tokenizer, names):
    result = {}
    for name in names:
        for prefix in (" ", "", "\n    ", "\n"):
            ids = tokenizer.encode(prefix + name, add_special_tokens=False)
            if ids:
                result[name] = ids[0]
                break
    return result

# ── Helper: find probe position in token stream ───────────────────────────────
def find_probe_positions(token_ids_seq, tokenizer, target_names, probe_at="pre"):
    parts = [tokenizer.decode([t]) for t in token_ids_seq]
    cumulative, running = [], ""
    for p in parts:
        running += p
        cumulative.append(running)
    positions = {}
    for name in target_names:
        pos = None
        for i, (before, at) in enumerate(zip([""] + cumulative[:-1], cumulative)):
            if name not in before and name in at:
                pos = max(0, i - 1) if probe_at == "pre" else i
                break
        positions[name] = pos
    return positions

all_names = [n for m in mutations for n in (m["original"], m["mutated"])]
name_to_tid = get_first_token_ids(tokenizer, all_names)
probe_positions = find_probe_positions(
    full_ids[response_start:], tokenizer, all_names, probe_at=PROBE_AT
)

print("Token ID mapping:")
for name, tid in name_to_tid.items():
    pos = probe_positions.get(name)
    found = f"response tok {pos}" if pos is not None else "NOT FOUND"
    print(f"  {name!r:25s} → token_id={tid:6d}  ({tokenizer.decode([tid])!r})  [{found}]")

In [ ]:
# ── Logit lens helper ─────────────────────────────────────────────────────────
W_U = model.lm_head.weight.detach().float()   # (vocab_size, hidden_size)
has_ln = hasattr(model, "model") and hasattr(model.model, "norm")

def logit_lens_at_position(hidden_states, position, token_ids):
    """
    For each layer, project hidden state at `position` through W_U → softmax.
    Returns list of {layer, probs, logits} dicts.
    """
    results = []
    for l, hs in enumerate(hidden_states):
        h = hs[0, position, :].float().cpu()
        if has_ln and l < len(hidden_states) - 1:
            with torch.no_grad():
                h = model.model.norm(h.to(next(model.parameters()).device).unsqueeze(0)).squeeze(0).float().cpu()
        logits = W_U.cpu() @ h
        log_probs = F.log_softmax(logits, dim=-1)
        results.append({
            "layer":  l,
            "probs":  {tid: math.exp(log_probs[tid].item()) for tid in token_ids},
            "logits": {tid: logits[tid].item()              for tid in token_ids},
        })
    return results

print("Logit lens helper ready.")

In [ ]:
# ── Run logit lens for each mutation ─────────────────────────────────────────
all_results = []

for mutation in mutations:
    orig = mutation["original"]
    mut  = mutation["mutated"]

    orig_pos = probe_positions.get(orig)
    mut_pos  = probe_positions.get(mut)
    winner_name, winner_pos = (mut, mut_pos) if mut_pos is not None else (orig, orig_pos)
    outcome = "doc" if mut_pos is not None else "memory"

    if winner_pos is None:
        print(f"  {orig}→{mut}: neither found in output, skipping")
        continue

    abs_pos   = response_start + winner_pos
    abs_pos   = min(abs_pos, len(full_ids) - 1)
    win_start = max(0, abs_pos - WINDOW + 1)
    window    = full_ids[win_start : abs_pos + 1]
    pos_in_w  = len(window) - 1

    print(f"  [{('DOC WON' if outcome=='doc' else 'MEMORY WON'):10s}]  {orig!r} ↔ {mut!r}"
          f"  (window {win_start}:{abs_pos+1}, {len(window)} tokens)", flush=True)

    win_tensor = torch.tensor([window], device=device)
    with torch.no_grad():
        out = model(win_tensor, output_hidden_states=True)
    hidden_states = out.hidden_states
    del out, win_tensor
    if device.type == "cuda":
        torch.cuda.empty_cache()

    orig_tid = name_to_tid.get(orig)
    mut_tid  = name_to_tid.get(mut)
    tids     = [t for t in [orig_tid, mut_tid] if t is not None]

    layer_data = logit_lens_at_position(hidden_states, pos_in_w, tids)
    del hidden_states

    # Print table
    print(f"  {'Layer':>8}  {'P('+orig+')':>22}  {'P('+mut+')':>22}  {'Leader':>12}  {'Gap(nats)':>10}")
    print(f"  {'-'*82}")
    prev_leader, flip_layer = None, None
    for ld in layer_data:
        l      = ld["layer"]
        p_o    = ld["probs"].get(orig_tid, 0.0) if orig_tid else 0.0
        p_m    = ld["probs"].get(mut_tid,  0.0) if mut_tid  else 0.0
        leader = orig if p_o >= p_m else mut
        gap    = abs(math.log(p_o + 1e-30) - math.log(p_m + 1e-30))
        bar_o  = int(p_o / max(p_o + p_m, 1e-9) * 20)
        bar    = "O" * bar_o + "M" * (20 - bar_o)
        pct    = l / max(n_layers, 1) * 100
        label  = ("early" if pct < 33 else "mid" if pct < 66 else "late ") + f" L{l:02d}"
        if prev_leader is not None and leader != prev_leader and flip_layer is None:
            flip_layer = l
            print(f"  {'':>8}  ← LEAD FLIPS HERE at layer {l}")
        print(f"  {label:>10}  {p_o:>22.6f}  {p_m:>22.6f}  {leader:>12s}  {gap:>10.2f}  [{bar}]")
        prev_leader = leader
    print()

    all_results.append({
        "mutation": mutation,
        "outcome": outcome,
        "probe_pos": abs_pos,
        "flip_layer": flip_layer,
        "layers": layer_data,
    })

In [ ]:
# ── Plot: P(orig) vs P(mut) across layers ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

n_mutations = len(all_results)
if n_mutations == 0:
    print("No results to plot.")
else:
    fig, axes = plt.subplots(1, n_mutations, figsize=(5 * n_mutations, 4), sharey=False)
    if n_mutations == 1:
        axes = [axes]

    for ax, res in zip(axes, all_results):
        orig = res["mutation"]["original"]
        mut  = res["mutation"]["mutated"]
        orig_tid = name_to_tid.get(orig)
        mut_tid  = name_to_tid.get(mut)

        layers  = [ld["layer"] for ld in res["layers"]]
        p_origs = [ld["probs"].get(orig_tid, 0.0) if orig_tid else 0.0 for ld in res["layers"]]
        p_muts  = [ld["probs"].get(mut_tid,  0.0) if mut_tid  else 0.0 for ld in res["layers"]]

        ax.plot(layers, p_origs, label=f"P({orig}) [memory]", color="tomato",     linewidth=2)
        ax.plot(layers, p_muts,  label=f"P({mut}) [doc]",     color="steelblue",  linewidth=2)

        if res["flip_layer"] is not None:
            ax.axvline(res["flip_layer"], color="gray", linestyle="--", alpha=0.7,
                       label=f"flip @ L{res['flip_layer']}")

        outcome_label = "DOC WON" if res["outcome"] == "doc" else "MEMORY WON"
        ax.set_title(f"{orig} → {mut}\n[{outcome_label}]", fontsize=10)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Probability")
        ax.legend(fontsize=8)
        ax.set_xlim(0, n_layers)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"Logit Lens — {DATASET} / {MODEL_ID.split('/')[-1]}", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"logit_lens_{DATASET}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved logit_lens_{DATASET}.png")

In [ ]:
# ── Save raw results ──────────────────────────────────────────────────────────
import json
out = {
    "dataset": DATASET,
    "model": MODEL_ID,
    "probe_at": PROBE_AT,
    "response_text": response_text,
    "mutations": mutations,
    "results": [
        {
            "mutation":    r["mutation"],
            "outcome":     r["outcome"],
            "probe_pos":   r["probe_pos"],
            "flip_layer":  r["flip_layer"],
            "layers": [
                {
                    "layer":  ld["layer"],
                    "probs":  {str(k): v for k, v in ld["probs"].items()},
                    "logits": {str(k): v for k, v in ld["logits"].items()},
                }
                for ld in r["layers"]
            ],
        }
        for r in all_results
    ],
}
fname = f"logit_lens_{DATASET}.json"
with open(fname, "w") as f:
    json.dump(out, f, indent=2)
print(f"Saved {fname}")